# NB-figs — generate the remaining paper figures

Produces, into `/kaggle/working/figs`:
- **fig_qualitative.pdf** — predictions across all 3 regimes, ours vs U-Net(no curric). The headline visual.
- **fig_dataset.pdf** — S1 / S2 / label overview for a few regions.
- **fig_curriculum.pdf** — the exact schedule, read from `lgmf.curriculum` (not hand-drawn).
- **fig_failure.pdf** — honest failure cases (optional but reviewers reward it).

All vector (PDF+SVG) + 300-dpi PNG, one shared colormap.

**Attach:** `lgmf_c` (v7), Sen1Floods11, `lgmf-index`, `lgmf-segf`, `lgmf-baselines`.
GPU on; Internet **off** (weights are loaded from checkpoints). ~10 min.


In [ ]:
import sys, os, glob, json, dataclasses, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v7"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch, rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.model import build_model
from lgmf.baselines import build_baseline, build_gated_unet
from lgmf.curriculum import regime_probs
print("lgmf", lgmf.__version__)

idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIG = "/kaggle/working/figs"; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.size": 8, "font.family": "DejaVu Sans"})

# ---- one shared palette for every figure ----
COL_WATER = (0.11, 0.42, 0.69, 1.0)   # blue  - predicted/true water
COL_FP    = (0.85, 0.37, 0.00, 1.0)   # orange- false positive
COL_FN    = (0.80, 0.47, 0.65, 1.0)   # pink  - false negative
def save(fig, name):
    for ext, dpi in [("png", 300), ("pdf", None), ("svg", None)]:
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=dpi, bbox_inches="tight")
    print("saved", name)

## Load models (ours + the no-curriculum baseline)

In [ ]:
FIELDS = {f.name for f in dataclasses.fields(LGMFConfig)}
def build_from_ckpt(ck):
    if ck.get("gated"):
        kw = {k: v for k, v in ck["gated"].items() if k != "out_ch"}
        m = build_gated_unet(pretrained=False, out_ch=ck["gated"].get("out_ch", 8), **kw)
    elif ck.get("baseline"):
        kw = {k: v for k, v in ck["baseline"].items() if k != "in_ch"}
        m = build_baseline(pretrained=False, in_ch=ck["baseline"].get("in_ch", 8), **kw)
    else:
        c = LGMFConfig(**{k: v for k, v in ck["cfg"].items() if k in FIELDS})
        c.s2_bands = tuple(c.s2_bands); c.encoder_pretrained = False
        m = build_model(c)
    m.load_state_dict(ck.get("ema") or ck["model"]); return m.to(device).eval()

def find_ckpt(cfgname, seed="seed42"):
    hits = [p for p in glob.glob("/kaggle/input/**/best.pt", recursive=True)
            if f"/{cfgname}/" in p and seed in p]
    if not hits:
        hits = [p for p in glob.glob("/kaggle/input/**/best.pt", recursive=True)
                if f"/{cfgname}/" in p]
    assert hits, f"no checkpoint for {cfgname}"
    return hits[0]

MODELS = {}
for name, label in [("segformer_concat", "Ours (SegFormer + curric + consist)"),
                    ("unet_drop",        "U-Net + curric"),
                    ("unet_naive",       "U-Net (no curric)")]:
    try:
        MODELS[name] = (label, build_from_ckpt(torch.load(find_ckpt(name), map_location=device, weights_only=False)))
        print("loaded", name)
    except AssertionError as e:
        print("SKIP", name, e)

## Test data + prediction helper

In [ ]:
base = LGMFConfig()
base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]
base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]
base.num_workers = 0
test_items = json.load(open(f"{idx}/test_index.json"))
ds = Sen1Floods11(base, f"{idx}/test_index.json", augment=False)
print(f"{len(ds)} test chips")

REGIMES = {"both": [1., 1.], "s2_only": [0., 1.], "s1_only": [1., 0.]}

@torch.no_grad()
def predict(model, sample, regime):
    s1 = sample["s1"].unsqueeze(0).to(device)
    s2 = sample["s2"].unsqueeze(0).to(device)
    p = torch.tensor([REGIMES[regime]], device=device, dtype=torch.float32)
    s1 = s1 * p[0, 0]; s2 = s2 * p[0, 1]
    logit = model(s1, s2, p)["logits"]
    return (torch.sigmoid(logit)[0, 0].cpu().numpy() > 0.5).astype(np.float32)

def s2_rgb(s2):
    rgb = s2[[2, 1, 0]].numpy().transpose(1, 2, 0)      # subset order B4,B3,B2
    lo, hi = np.percentile(rgb, (2, 98))
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

def s1_gray(s1):
    v = s1[0].numpy()
    lo, hi = np.percentile(v, (2, 98))
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1)

def overlay(base_img, mask, color=COL_WATER, alpha=0.55):
    out = np.dstack([base_img]*3) if base_img.ndim == 2 else base_img.copy()
    m = mask > 0.5
    for c in range(3):
        out[..., c] = np.where(m, (1-alpha)*out[..., c] + alpha*color[c], out[..., c])
    return out

def err_map(pred, gt, valid):
    h, w = pred.shape; out = np.ones((h, w, 3))
    v = valid > 0.5
    tp = (pred > .5) & (gt > .5) & v
    fp = (pred > .5) & (gt <= .5) & v
    fn = (pred <= .5) & (gt > .5) & v
    for c in range(3):
        out[..., c] = np.where(tp, COL_WATER[c], out[..., c])
        out[..., c] = np.where(fp, COL_FP[c], out[..., c])
        out[..., c] = np.where(fn, COL_FN[c], out[..., c])
    return out

def iou(pred, gt, valid):
    v = valid > .5; p = (pred > .5) & v; g = (gt > .5) & v
    u = (p | g).sum()
    return float((p & g).sum() / u) if u else float("nan")

## Pick chips for the qualitative figure

Chooses test chips with a healthy amount of water and low no-data, then keeps a
spread of regions. **Edit `CHOSEN` by hand** once you see the candidates — pick
visually clear examples rather than whatever sorts first.


In [ ]:
cands = []
for i in range(len(ds)):
    s = ds[i]
    valid = s["valid"].numpy(); lbl = s["label"].numpy()
    vfrac = valid.mean(); wfrac = (lbl * valid).sum() / max(valid.sum(), 1)
    if vfrac > 0.9 and 0.12 < wfrac < 0.6:
        cands.append((i, test_items[i]["id"], round(float(wfrac), 3)))
print(f"{len(cands)} candidate chips (>90% valid, 12-60% water):")
for c in cands[:25]: print("   ", c)

# one per region for variety
seen, CHOSEN = set(), []
for i, cid, w in cands:
    reg = cid.split("_")[0]
    if reg not in seen:
        seen.add(reg); CHOSEN.append(i)
    if len(CHOSEN) == 4: break
print("\nCHOSEN:", [(i, test_items[i]['id']) for i in CHOSEN])

## Figure — qualitative predictions across regimes (headline)

In [ ]:
main_name = "segformer_concat"
rows = CHOSEN[:3]
cols = ["S2 (RGB)", "S1 (VV)", "Ground truth",
        "Ours\nboth", "Ours\noptical-only", "Ours\nSAR-only", "No curric.\nSAR-only"]
fig, axes = plt.subplots(len(rows), len(cols),
                         figsize=(1.55*len(cols), 1.62*len(rows)))
axes = np.atleast_2d(axes)
for r, i in enumerate(rows):
    s = ds[i]; gt = s["label"].numpy(); valid = s["valid"].numpy()
    rgb, gray = s2_rgb(s["s2"]), s1_gray(s["s1"])
    panels = [rgb, np.dstack([gray]*3), overlay(rgb, gt)]
    titles = ["", "", ""]
    for reg in ["both", "s2_only", "s1_only"]:
        p = predict(MODELS[main_name][1], s, reg)
        panels.append(err_map(p, gt, valid)); titles.append(f"IoU {iou(p,gt,valid):.2f}")
    if "unet_naive" in MODELS:
        p = predict(MODELS["unet_naive"][1], s, "s1_only")
        panels.append(err_map(p, gt, valid)); titles.append(f"IoU {iou(p,gt,valid):.2f}")
    else:
        panels.append(np.ones_like(rgb)); titles.append("")
    for c in range(len(cols)):
        ax = axes[r, c]; ax.imshow(panels[c]); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_linewidth(0.4); sp.set_color("#999")
        if r == 0: ax.set_title(cols[c], fontsize=7)
        if titles[c]:
            ax.text(0.5, -0.06, titles[c], transform=ax.transAxes,
                    ha="center", va="top", fontsize=6.2)
    axes[r, 0].set_ylabel(test_items[i]["id"].split("_")[0], fontsize=7)
handles = [Patch(fc=COL_WATER, label="true positive"),
           Patch(fc=COL_FP, label="false positive"),
           Patch(fc=COL_FN, label="false negative")]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=6.8,
           frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.subplots_adjust(wspace=0.05, hspace=0.22)
save(fig, "fig_qualitative"); plt.show()

## Figure — dataset overview

In [ ]:
rows2 = CHOSEN[:3]
fig, axes = plt.subplots(len(rows2), 3, figsize=(5.4, 1.85*len(rows2)))
axes = np.atleast_2d(axes)
for r, i in enumerate(rows2):
    s = ds[i]; rgb, gray = s2_rgb(s["s2"]), s1_gray(s["s1"])
    gt = s["label"].numpy()
    for c, (img, t) in enumerate([(rgb, "Sentinel-2 (RGB)"),
                                  (np.dstack([gray]*3), "Sentinel-1 (VV)"),
                                  (overlay(rgb, gt), "flood label")]):
        ax = axes[r, c]; ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_linewidth(0.4); sp.set_color("#999")
        if r == 0: ax.set_title(t, fontsize=7.5)
    axes[r, 0].set_ylabel(test_items[i]["id"].split("_")[0], fontsize=7)
fig.subplots_adjust(wspace=0.04, hspace=0.06)
save(fig, "fig_dataset"); plt.show()

## Figure — curriculum schedule (exact, from lgmf.curriculum)

In [ ]:
cfg = LGMFConfig(); cfg.epochs = 45; cfg.use_curriculum = True
print(f"epochs={cfg.epochs} warmup={cfg.warmup_epochs} final={cfg.final_regime_probs}")
E = np.arange(cfg.epochs)
P = np.array([regime_probs(cfg, e) for e in E])
fig, ax = plt.subplots(figsize=(3.5, 2.7))
for k, (lbl, col) in enumerate([("both", "#8a8778"), ("SAR-only", "#1d6fb8"),
                                ("optical-only", "#0f6e56")]):
    ax.plot(E, P[:, k], color=col, lw=1.8, label=lbl)
if cfg.warmup_epochs:
    ax.axvspan(0, cfg.warmup_epochs, color="#00000010", lw=0)
    ax.text(cfg.warmup_epochs/2, .5, "warm-up", rotation=90, fontsize=6,
            ha="center", va="center", color="#555")
ax.set_xlabel("epoch", fontsize=7.5); ax.set_ylabel("sampling probability", fontsize=7.5)
ax.set_xlim(0, cfg.epochs-1); ax.set_ylim(0, 1.05); ax.tick_params(labelsize=6.5)
ax.grid(alpha=.25, lw=.5); ax.legend(fontsize=6.5, frameon=False, loc="center right")
save(fig, "fig_curriculum"); plt.show()
for e in [0, cfg.warmup_epochs, cfg.epochs//2, cfg.epochs-1]:
    p = regime_probs(cfg, e); print(f"  ep{e:3d}: both {p[0]:.3f} s1 {p[1]:.3f} s2 {p[2]:.3f}")

## Figure — failure cases (optional, but reviewers reward honesty)

In [ ]:
scored = []
for i in range(len(ds)):
    s = ds[i]; gt = s["label"].numpy(); valid = s["valid"].numpy()
    if valid.mean() < 0.8: continue
    p = predict(MODELS[main_name][1], s, "s1_only")
    scored.append((iou(p, gt, valid), i))
scored.sort()
worst = [i for _, i in scored[:3]]
print("worst SAR-only chips:", [(test_items[i]['id'], round(v,3)) for v, i in scored[:3]])

fig, axes = plt.subplots(len(worst), 4, figsize=(6.6, 1.72*len(worst)))
axes = np.atleast_2d(axes)
for r, i in enumerate(worst):
    s = ds[i]; gt = s["label"].numpy(); valid = s["valid"].numpy()
    rgb, gray = s2_rgb(s["s2"]), s1_gray(s["s1"])
    pb = predict(MODELS[main_name][1], s, "both")
    p1 = predict(MODELS[main_name][1], s, "s1_only")
    panels = [(rgb, ""), (np.dstack([gray]*3), ""),
              (err_map(pb, gt, valid), f"IoU {iou(pb,gt,valid):.2f}"),
              (err_map(p1, gt, valid), f"IoU {iou(p1,gt,valid):.2f}")]
    for c, (img, t) in enumerate(panels):
        ax = axes[r, c]; ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_linewidth(0.4); sp.set_color("#999")
        if r == 0: ax.set_title(["S2 (RGB)","S1 (VV)","both","SAR-only"][c], fontsize=7.5)
        if t: ax.text(.5, -.06, t, transform=ax.transAxes, ha="center", va="top", fontsize=6.2)
    axes[r, 0].set_ylabel(test_items[i]["id"].split("_")[0], fontsize=7)
fig.subplots_adjust(wspace=.04, hspace=.2)
save(fig, "fig_failure"); plt.show()

## Done

`/kaggle/working/figs` now holds fig_qualitative, fig_dataset, fig_curriculum,
fig_failure (+ degradation and pareto from the results notebook) in PDF/SVG/PNG.

**Save the notebook output as a dataset before the session ends.**
If a chip looks unconvincing, edit `CHOSEN` in the selection cell and re-run the
qualitative cell only.
